# MCU-A Bandgap Trim Prediction

This notebook runs the same training and evaluation code used from VS Code. It uses the supplied synthetic dataset and keeps whole wafers out of training when evaluating. Predictions are demo candidate codes only; the app does not program fuses.


## Upload the project ZIP

Download and unzip nothing locally for this step. In the next cell, choose the project ZIP file from your computer. The notebook extracts it into the Colab runtime, installs its dependencies, and invokes train_model.py.


In [ ]:
from google.colab import files
from pathlib import Path
from zipfile import ZipFile

uploaded = files.upload()
zip_name = next((name for name in uploaded if name.lower().endswith(".zip")), None)
if zip_name is None:
    raise ValueError("Upload the trim-fuse-ml-demo project ZIP.")

project_dir = Path("/content/trim-fuse-ml-demo")
project_dir.mkdir(parents=True, exist_ok=True)
with ZipFile(zip_name) as archive:
    root = project_dir.resolve()
    for member in archive.infolist():
        destination = (project_dir / member.filename).resolve()
        if destination != root and root not in destination.parents:
            raise ValueError(f"Unsafe path in archive: {member.filename}")
    archive.extractall(project_dir.parent)

print("Project folder:", project_dir)
print("Files:", len(list(project_dir.rglob("*"))))


## Install the project dependencies

The CSV, training script, Streamlit UI, and smoke checks are in the uploaded project folder.


In [ ]:
%cd /content/trim-fuse-ml-demo
!pip install -q -r requirements.txt


## Train, select, and evaluate

The script checks the data, holds out complete wafers, compares models with grouped folds on training wafers, fits the chosen pipeline, evaluates once on held-out wafers, and writes artifacts.


In [ ]:
!python train_model.py


## Inspect the quality report and metrics

Use the within-one-code rate as the main predict-then-verify metric. Exact accuracy, MAE, and RMSE give complementary views. The reported data is synthetic, and iteration savings are an estimate until measured on tester hardware.


In [ ]:
import json
from pathlib import Path

metrics = json.loads(Path("artifacts/metrics.json").read_text())
quality = json.loads(Path("artifacts/data_quality_report.json").read_text())
print("Rows / columns:", quality["raw_rows"], "/", quality["raw_columns"])
print("Missing cells:", quality["source_missing_cells"])
print("Exact duplicate rows removed:", quality["exact_duplicate_rows_removed"])
print("Duplicate die IDs:", quality["duplicate_die_ids"])
print("MCU-A rows / wafers:", quality["mcu_a_rows"], "/", quality["mcu_a_wafer_count"])
print("Selected model:", metrics["selected_model"])
print("Holdout metrics:", json.dumps(metrics["holdout_metrics"], indent=2))
print("Conventional iterations:", metrics["conventional_flow_on_test_wafers"]["mean_iterations"])
print("Estimated candidate-flow iterations:", metrics["predict_then_verify_estimate"]["mean_iterations"])


## Run smoke checks

These checks verify quality, wafer isolation, and legal inference output. A live app can be run locally in VS Code with python -m streamlit run app.py.


In [ ]:
!python -m unittest discover -s tests -v
